# Historical human benchmark — E2025

This notebook compares the frozen Phase 7 `FULL_DYNAMIC_STOCHASTIC` point-in-time replay for E2025 (the 2025–26 season) with the published BasketStories Private League Top-10 standings. It does not train models or alter optimizer logic. Human benchmark data is fetched only from the cited BasketStories pages and is not joined to prediction/training data.

In [21]:
from __future__ import annotations

from html.parser import HTMLParser
from pathlib import Path
import json
import re
from urllib.request import Request, urlopen

import duckdb
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKTEST_PATH = PROJECT_ROOT / "data/derived/phase7/research/e2025_strategy_backtest.parquet"
PREDICTIONS_PATH = PROJECT_ROOT / "data/derived/phase7/research/e2025_full_market_predictions.parquet"
CUTOFF_AUDIT_PATH = PROJECT_ROOT / "data/derived/phase7/research/e2025_replay_cutoff_audit.json"
DATABASE_PATH = PROJECT_ROOT / "data/db/euroleague.duckdb"
STRATEGY = "FULL_DYNAMIC_STOCHASTIC"
RETRIEVED_AT = "2026-09-04"
BASE_URL = "https://www.basketstories.net/article.php?p=private_league_post_"

ROUND_POST_IDS = {
    1: 5564, 2: 5576, 3: 5591, 4: 5606, 5: 5616, 6: 5633,
    7: 5649, 8: 5660, 9: 5675, 10: 5689, 11: 5701, 12: 5713,
    13: 5728, 14: 5741, 15: 5756, 16: 5769, 17: 5781, 18: 5796,
    19: 5808, 20: 5821, 21: 5827, 22: 5841, 23: 5854, 24: 5864,
    25: 5876, 26: 5889, 27: 5901, 28: 5914, 29: 5926, 30: 5936,
    31: 5948, 32: 5962, 33: 5973, 34: 5983, 35: 5994,
    36: 6025, 37: 6025, 38: 6025,
}

assert sorted(ROUND_POST_IDS) == list(range(1, 39))
print(f"Configured {len(ROUND_POST_IDS)} round-to-source mappings.")

Configured 38 round-to-source mappings.


In [15]:
class StandingsTableParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.tables = []
        self._in_target = False
        self._rows = []
        self._row = None
        self._cell = None

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "table" and "table_yes" in attrs.get("class", "").split():
            self._in_target = True
            self._rows = []
        elif self._in_target and tag == "tr":
            self._row = []
        elif self._in_target and tag == "td":
            self._cell = []

    def handle_data(self, data):
        if self._in_target and self._cell is not None:
            self._cell.append(data)

    def handle_endtag(self, tag):
        if self._in_target and tag == "td" and self._cell is not None:
            self._row.append("".join(self._cell).strip())
            self._cell = None
        elif self._in_target and tag == "tr" and self._row is not None:
            if self._row:
                self._rows.append(self._row)
            self._row = None
        elif self._in_target and tag == "table":
            # Two source pages omit the final </tr>; retain that complete row.
            if self._row:
                self._rows.append(self._row)
                self._row = None
            self.tables.append(pd.DataFrame(self._rows))
            self._in_target = False
            self._rows = []


def parse_standings_tables(html: str) -> list[pd.DataFrame]:
    # Two posts contain the literal manager name "Kostas <Kechris", where the
    # stray angle bracket is malformed markup. Normalize that source typo.
    html = html.replace("Kostas <Kechris", "Kostas Kechris")
    parser = StandingsTableParser()
    parser.feed(html)
    return parser.tables


def fetch_article(post_id: int) -> tuple[str, str, str]:
    url = f"{BASE_URL}{post_id}"
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=60) as response:
        html = response.read().decode("utf-8-sig")
    date_match = re.search(r'<div id="post_date">[^:]+:\s*([^<]+)', html)
    if not date_match:
        raise ValueError(f"Publication date not found for {url}")
    return url, date_match.group(1).strip(), html


def standardize_round_table(table: pd.DataFrame, matchday: int) -> pd.DataFrame:
    if table.shape[1] not in (4, 5):
        raise ValueError(f"Unexpected round table width for MD{matchday}: {table.shape}")
    columns = ["round_rank", "team_name", "manager_name", "round_score"]
    if table.shape[1] == 5:
        columns.append("manager_cumulative_score")
    table = table.copy()
    table.columns = columns
    if "manager_cumulative_score" not in table:
        table["manager_cumulative_score"] = table["round_score"]
    table.insert(0, "matchday", matchday)
    return table


def standardize_cumulative_table(table: pd.DataFrame, matchday: int) -> pd.DataFrame:
    if table.shape[1] != 4:
        raise ValueError(f"Unexpected cumulative table width for MD{matchday}: {table.shape}")
    table = table.copy()
    table.columns = ["cumulative_rank", "team_name", "manager_name", "cumulative_score"]
    table.insert(0, "matchday", matchday)
    return table


article_cache = {}
for post_id in sorted(set(ROUND_POST_IDS.values())):
    article_cache[post_id] = fetch_article(post_id)

round_frames = []
cumulative_frames = []
provenance_rows = []

for matchday, post_id in ROUND_POST_IDS.items():
    url, published_at, html = article_cache[post_id]
    tables = parse_standings_tables(html)

    if matchday <= 35:
        round_table = tables[0]
        cumulative_table = tables[1] if len(tables) > 1 else tables[0]
    else:
        # The regular-season winners article contains final cumulative standings,
        # followed by round tables for MD38, MD37, and MD36.
        round_table = tables[{38: 1, 37: 2, 36: 3}[matchday]]
        cumulative_table = tables[0] if matchday == 38 else None

    round_frame = standardize_round_table(round_table, matchday)
    round_frame["source_url"] = url
    round_frames.append(round_frame)

    if cumulative_table is not None:
        cumulative_frame = standardize_cumulative_table(cumulative_table, matchday)
        cumulative_frame["source_url"] = url
        cumulative_frames.append(cumulative_frame)

    provenance_rows.append({
        "matchday": matchday,
        "post_id": f"private_league_post_{post_id}",
        "source_url": url,
        "published_at": published_at,
        "retrieved_at": RETRIEVED_AT,
        "round_rows": len(round_frame),
        "cumulative_top10_available": cumulative_table is not None,
    })

human_round = pd.concat(round_frames, ignore_index=True)
human_cumulative = pd.concat(cumulative_frames, ignore_index=True)
provenance = pd.DataFrame(provenance_rows)

for column in ["round_rank", "round_score", "manager_cumulative_score"]:
    human_round[column] = pd.to_numeric(human_round[column], errors="raise")
for column in ["cumulative_rank", "cumulative_score"]:
    human_cumulative[column] = pd.to_numeric(human_cumulative[column], errors="raise")

assert len(human_round) == 380
assert human_round.groupby("matchday").size().eq(10).all()
assert human_round["round_score"].between(0, 400).all()
assert human_round.groupby("matchday")["round_rank"].nunique().eq(10).all()
# The MD12 source itself numbers its ten published rows 2–11; preserve that provenance anomaly.
assert human_round.loc[human_round["matchday"].eq(12), "round_rank"].tolist() == list(range(2, 12))
assert set(human_cumulative["matchday"]) == set(range(1, 36)) | {38}
assert human_cumulative.groupby("matchday").size().eq(10).all()
assert all(
    ranks == list(range(1, 11))
    for ranks in human_cumulative.groupby("matchday")["cumulative_rank"].apply(list)
)

print(f"Parsed {len(human_round)} round rows across {human_round.matchday.nunique()} matchdays.")
print(f"Parsed cumulative Top-10 tables for {human_cumulative.matchday.nunique()} matchdays; MD36–37 were not published.")

Parsed 380 round rows across 38 matchdays.
Parsed cumulative Top-10 tables for 36 matchdays; MD36–37 were not published.


In [26]:
with duckdb.connect() as connection:
    backtest = connection.execute(
        "SELECT * FROM read_parquet(?)", [str(BACKTEST_PATH)]
    ).fetchdf()
ai = (
    backtest.loc[backtest["strategy"].eq(STRATEGY)]
    .sort_values("matchday")
    .reset_index(drop=True)
    .copy()
)
assert ai["matchday"].tolist() == list(range(1, 39))
assert ai["season"].eq("E2025").all()
ai["ai_round_score"] = pd.to_numeric(ai["actual_final_score"], errors="raise")
ai["ai_cumulative_score"] = ai["ai_round_score"].cumsum()

with duckdb.connect() as connection:
    player_names = connection.execute(
        "SELECT DISTINCT player_id AS entity_id, name FROM read_parquet(?)",
        [str(PREDICTIONS_PATH)],
    ).fetchdf()
with duckdb.connect(str(DATABASE_PATH), read_only=True) as connection:
    coach_names = connection.execute(
        """
        SELECT fantasy_entity_id AS entity_id, display_name AS name
        FROM fantasy_entities
        WHERE upper(entity_type) = 'COACH'
        """
    ).fetchdf()

entity_name_map = dict(
    pd.concat([player_names, coach_names], ignore_index=True)
    .dropna()
    .drop_duplicates("entity_id")
    .itertuples(index=False, name=None)
)

def names_for_pipe(ids: str) -> str:
    return " | ".join(entity_name_map.get(entity_id, entity_id) for entity_id in ids.split("|"))

ai["roster_names"] = ai["roster"].map(names_for_pipe)
ai["starters_names"] = ai["starters"].map(names_for_pipe)
ai["captain_name"] = ai["captain"].map(entity_name_map).fillna(ai["captain"])
ai["sixth_man_name"] = ai["sixth_man"].map(entity_name_map).fillna(ai["sixth_man"])
ai["coach_name"] = ai["coach_id"].map(entity_name_map).fillna(ai["coach_id"])

round_summary = (
    human_round.groupby("matchday")
    .agg(
        best_human_round_score=("round_score", "max"),
        top5_average=("round_score", lambda s: s.nlargest(5).mean()),
        top10_average=("round_score", "mean"),
    )
    .reset_index()
)
round_winners = (
    human_round.sort_values(
        ["matchday", "round_score"], ascending=[True, False]
    )
    .groupby("matchday", as_index=False)
    .head(1)[["matchday", "team_name", "manager_name"]]
    .rename(columns={
        "team_name": "best_human_team",
        "manager_name": "best_human_manager",
    })
)
round_summary = round_summary.merge(
    round_winners, on="matchday", how="left", validate="one_to_one"
)

cumulative_summary = (
    human_cumulative.loc[human_cumulative["cumulative_rank"].eq(1), [
        "matchday", "team_name", "manager_name", "cumulative_score"
    ]]
    .rename(columns={
        "team_name": "human_leader_team",
        "manager_name": "human_leader_manager",
        "cumulative_score": "human_leader_cumulative",
    })
)

benchmark = ai[[
    "matchday", "ai_round_score", "ai_cumulative_score", "total_credits",
    "available_budget", "bank_after", "trades", "trade_limit", "captain",
    "sixth_man", "starters", "roster", "coach_id", "captain_name",
    "sixth_man_name", "starters_names", "roster_names", "coach_name",
]].merge(round_summary, on="matchday", validate="one_to_one")
benchmark = benchmark.merge(cumulative_summary, on="matchday", how="left", validate="one_to_one")
benchmark["gap_vs_top10_average"] = benchmark["ai_round_score"] - benchmark["top10_average"]
benchmark["gap_vs_top5_average"] = benchmark["ai_round_score"] - benchmark["top5_average"]
benchmark["gap_vs_best_human"] = benchmark["ai_round_score"] - benchmark["best_human_round_score"]
benchmark["cumulative_gap_vs_leader"] = (
    benchmark["ai_cumulative_score"] - benchmark["human_leader_cumulative"]
)

cutoff_audit = json.loads(CUTOFF_AUDIT_PATH.read_text())
audit_rows = pd.DataFrame(cutoff_audit["matchdays"])
assert audit_rows["matchday"].tolist() == list(range(1, 39))
assert audit_rows["max_feature_time_strictly_before_cutoff"].all()
assert audit_rows["conditional_on_playing_predictions"].all()
assert audit_rows["actual_dnp_scored_zero_only_at_replay"].all()
assert np.allclose(
    benchmark["ai_cumulative_score"].to_numpy(),
    benchmark["ai_round_score"].to_numpy().cumsum(),
)
assert benchmark["matchday"].equals(provenance["matchday"])

print("Alignment, point-in-time cutoff flags, and cumulative scoring checks passed.")
benchmark.head(3)

Alignment, point-in-time cutoff flags, and cumulative scoring checks passed.


,matchday,ai_round_score,ai_cumulative_score,total_credits,available_budget,bank_after,trades,trade_limit,captain,sixth_man,...,top10_average,best_human_team,best_human_manager,human_leader_team,human_leader_manager,human_leader_cumulative,gap_vs_top10_average,gap_vs_top5_average,gap_vs_best_human,cumulative_gap_vs_leader
0,1,112.25,112.25,100.0,100.0,0.0,11,NaN,66808e1c-a529-5c1f-9839-c8ba652c824c,81dd869f-825c-59a1-a8a8-e94fd03976c3,...,221.055,King of basketball,Christos Gouziotis,King of basketball,Christos Gouziotis,226.85,-108.805,-110.65,-114.60,-114.60
1,2,203.40,315.65,100.1,100.2,0.1,4,4.0,81dd869f-825c-59a1-a8a8-e94fd03976c3,faaf90f4-212d-577b-a113-944b5a99c143,...,199.325,nothin' but net,anastasios michairinas,nothin' but net,anastasios michairinas,389.90,4.075,1.90,-0.85,-74.25
2,3,140.20,455.85,103.4,103.6,0.2,4,4.0,15fa25ab-dd19-577b-a626-6623b9286e83,3b49d9cd-2a38-52f6-998f-1375262c45a9,...,221.095,Attila İlhan,HAŞMET ASİLKAN,disappointguard,d_ keram,585.90,-80.895,-82.37,-83.75,-130.05


In [27]:
final_ai = benchmark.loc[benchmark["matchday"].eq(38)].iloc[0]
final_top10 = human_cumulative.loc[human_cumulative["matchday"].eq(38)].sort_values("cumulative_rank")
published_above = int((final_top10["cumulative_score"] > final_ai["ai_cumulative_score"]).sum())
published_below = int((final_top10["cumulative_score"] < final_ai["ai_cumulative_score"]).sum())
equivalent_rank = published_above + 1 if published_below > 0 else None

overall = {
    "rounds": int(len(benchmark)),
    "ai_total": float(final_ai["ai_cumulative_score"]),
    "human_leader_total": float(final_ai["human_leader_cumulative"]),
    "final_gap_vs_leader": float(final_ai["cumulative_gap_vs_leader"]),
    "rounds_ai_beats_top10_average": int((benchmark["gap_vs_top10_average"] > 0).sum()),
    "rounds_ai_beats_top5_average": int((benchmark["gap_vs_top5_average"] > 0).sum()),
    "rounds_ai_beats_best_human": int((benchmark["gap_vs_best_human"] > 0).sum()),
    "mean_gap_vs_top10": float(benchmark["gap_vs_top10_average"].mean()),
    "mean_gap_vs_top5": float(benchmark["gap_vs_top5_average"].mean()),
    "mean_gap_vs_best": float(benchmark["gap_vs_best_human"].mean()),
    "published_final_equivalent_rank": equivalent_rank,
    "final_top10_low": float(final_top10["cumulative_score"].min()),
    "final_top10_high": float(final_top10["cumulative_score"].max()),
    "mean_budget_available": float(benchmark["available_budget"].mean()),
    "mean_credits_used": float(benchmark["total_credits"].mean()),
    "mean_unused_budget": float(benchmark["bank_after"].mean()),
    "total_recorded_trades_including_initial_roster": int(benchmark["trades"].sum()),
    "post_md1_transfers": int(benchmark.loc[benchmark["matchday"].gt(1), "trades"].sum()),
}
print(json.dumps(overall, indent=2))

display(
    benchmark.nsmallest(5, "gap_vs_top10_average")[[
        "matchday", "ai_round_score", "top10_average", "gap_vs_top10_average"
    ]].rename_axis("weakest")
)
display(
    benchmark.nlargest(5, "gap_vs_top10_average")[[
        "matchday", "ai_round_score", "top10_average", "gap_vs_top10_average"
    ]].rename_axis("strongest")
)
display(final_top10)

{
  "rounds": 38,
  "ai_total": 6416.349999999999,
  "human_leader_total": 6832.1,
  "final_gap_vs_leader": -415.7500000000009,
  "rounds_ai_beats_top10_average": 4,
  "rounds_ai_beats_top5_average": 2,
  "rounds_ai_beats_best_human": 0,
  "mean_gap_vs_top10": -48.12642105263158,
  "mean_gap_vs_top5": -52.37505263157894,
  "mean_gap_vs_best": -60.13631578947368,
  "published_final_equivalent_rank": 9,
  "final_top10_low": 6393.92,
  "final_top10_high": 6832.1,
  "mean_budget_available": 125.37368421052636,
  "mean_credits_used": 125.02894736842106,
  "mean_unused_budget": 0.3447368421052712,
  "total_recorded_trades_including_initial_roster": 172,
  "post_md1_transfers": 161
}


,matchday,ai_round_score,top10_average,gap_vs_top10_average
weakest,,,,
22,23,125.25,238.510,-113.260
0,1,112.25,221.055,-108.805
11,12,122.25,210.040,-87.790
4,5,143.20,226.220,-83.020
37,38,168.45,249.525,-81.075


,matchday,ai_round_score,top10_average,gap_vs_top10_average
strongest,,,,
16,17,243.00,233.275,9.725
1,2,203.40,199.325,4.075
29,30,196.70,193.205,3.495
20,21,229.60,227.548,2.052
9,10,220.75,225.215,-4.465


,matchday,cumulative_rank,team_name,manager_name,cumulative_score,source_url
350,38,1,wick2,Konstantinos mamalos,6832.10,https://www.basketstories.net/article.php?p=pr...
351,38,2,disappointguard,d _keram,6626.67,https://www.basketstories.net/article.php?p=pr...
352,38,3,Dalabc,Παναγιώτης Δαλαμπάκης,6582.84,https://www.basketstories.net/article.php?p=pr...
353,38,4,Apsurdistan,Deni Bjelobradić,6566.38,https://www.basketstories.net/article.php?p=pr...
354,38,5,Avengers,Andreas Fountas,6487.34,https://www.basketstories.net/article.php?p=pr...
355,38,6,Apache,Stelios Xenakis,6448.78,https://www.basketstories.net/article.php?p=pr...
356,38,7,Evolution Fitness,ALEXANDROS MOSHONAS,6429.97,https://www.basketstories.net/article.php?p=pr...
357,38,8,Blue Vayeros,Sakis Tzimas,6421.10,https://www.basketstories.net/article.php?p=pr...
358,38,9,v.p. nupezę vilkai,Vytautas Paulauskas,6411.66,https://www.basketstories.net/article.php?p=pr...
359,38,10,Btbuckets2,Bahattin Tütüncü,6393.92,https://www.basketstories.net/article.php?p=pr...


In [28]:
def markdown_table(frame: pd.DataFrame) -> str:
    def render(value):
        if pd.isna(value):
            return "—"
        if isinstance(value, (float, np.floating)):
            return f"{value:.2f}"
        return str(value).replace("|", chr(92) + "|")
    headers = [str(column) for column in frame.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    lines.extend(
        "| " + " | ".join(render(value) for value in row) + " |"
        for row in frame.itertuples(index=False, name=None)
    )
    return chr(10).join(lines)

round_report = benchmark[[
    "matchday", "ai_round_score", "best_human_round_score", "top5_average",
    "top10_average", "gap_vs_top10_average", "ai_cumulative_score",
    "human_leader_cumulative", "cumulative_gap_vs_leader",
]].round(2)

portfolio_report = benchmark[[
    "matchday", "ai_round_score", "ai_cumulative_score", "total_credits",
    "available_budget", "bank_after", "trades", "trade_limit", "captain",
    "captain_name", "sixth_man_name", "roster_names", "coach_name",
]].copy()

print(markdown_table(round_report))
print()
print("FINAL TOP 10")
print(markdown_table(final_top10))
print()
print("PROVENANCE")
print(markdown_table(provenance))

| matchday | ai_round_score | best_human_round_score | top5_average | top10_average | gap_vs_top10_average | ai_cumulative_score | human_leader_cumulative | cumulative_gap_vs_leader |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 112.25 | 226.85 | 222.90 | 221.06 | -108.80 | 112.25 | 226.85 | -114.60 |
| 2 | 203.40 | 204.25 | 201.50 | 199.32 | 4.08 | 315.65 | 389.90 | -74.25 |
| 3 | 140.20 | 223.95 | 222.57 | 221.09 | -80.89 | 455.85 | 585.90 | -130.05 |
| 4 | 151.65 | 217.80 | 210.21 | 204.91 | -53.26 | 607.50 | 723.20 | -115.70 |
| 5 | 143.20 | 233.15 | 229.79 | 226.22 | -83.02 | 750.70 | 934.45 | -183.75 |
| 6 | 179.50 | 218.85 | 215.24 | 214.02 | -34.53 | 930.20 | 1117.45 | -187.25 |
| 7 | 169.65 | 248.40 | 235.30 | 231.83 | -62.18 | 1099.85 | 1312.90 | -213.05 |
| 8 | 132.60 | 217.35 | 208.09 | 204.82 | -72.22 | 1232.45 | 1509.30 | -276.85 |
| 9 | 147.00 | 214.40 | 206.34 | 202.03 | -55.03 | 1379.45 | 1672.35 | -292.90 |
| 10 | 220.75 | 241.50 | 231.59 | 225.22 | -

In [29]:
EXPORTS = {
    "round": PROJECT_ROOT / "data/benchmarks/basketstories_private_league_e2025_round_standings.csv",
    "cumulative": PROJECT_ROOT / "data/benchmarks/basketstories_private_league_e2025_cumulative_standings.csv",
    "provenance": PROJECT_ROOT / "data/benchmarks/basketstories_private_league_e2025_provenance.csv",
    "comparison": PROJECT_ROOT / "data/benchmarks/e2025_ai_human_round_comparison.csv",
    "ledger": PROJECT_ROOT / "data/benchmarks/e2025_ai_replay_ledger.csv",
}
with duckdb.connect() as connection:
    exported = {
        name: connection.execute(
            "SELECT * FROM read_csv_auto(?, header=true)", [str(path)]
        ).fetchdf()
        for name, path in EXPORTS.items()
    }

assert len(exported["round"]) == 380
assert len(exported["cumulative"]) == 360
assert len(exported["provenance"]) == 38
assert len(exported["comparison"]) == 38
assert len(exported["ledger"]) == 38
assert exported["comparison"]["matchday"].tolist() == list(range(1, 39))
assert np.allclose(
    exported["comparison"]["ai_cumulative_score"],
    exported["comparison"]["ai_round_score"].cumsum(),
)
assert np.isclose(exported["comparison"].iloc[-1]["ai_cumulative_score"], 6416.35)
assert (PROJECT_ROOT / "reports/historical_human_fantasy_benchmark.md").is_file()

print("Export verification passed: standings parsing, round alignment, cutoff audit, and cumulative scoring.")

Export verification passed: standings parsing, round alignment, cutoff audit, and cumulative scoring.
